In [1]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

MODEL_NAME = "xlm-roberta-base"
CSV_PATH = "./Nazario_5_preprocessed_v2.csv"
TEXT_COL1 = "subject_cleaned_2"
TEXT_COL2 = "body_cleaned_2"
LABEL_COL = "intent_type"
MAX_LEN = 256
OUTPUT_DIR = "./xlmr_intent_model"


d:\VishnuMK\Projects\Email\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: module 'httpx' has no attribute 'RequestError'

In [3]:
df = pd.read_csv(CSV_PATH)

df[TEXT_COL1] = df[TEXT_COL1].fillna("")
df[TEXT_COL2] = df[TEXT_COL2].fillna("")
df[LABEL_COL] = df[LABEL_COL].fillna("unknown")

df["text"] = df[TEXT_COL1] + " [SEP] " + df[TEXT_COL2]
df = df[["text", LABEL_COL]].dropna()

label_list = sorted(df[LABEL_COL].unique().tolist())
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

df["labels"] = df[LABEL_COL].map(label2id)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["labels"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["labels"],
    random_state=42
)

train_ds = Dataset.from_pandas(train_df[["text", "labels"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["text", "labels"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["text", "labels"]].reset_index(drop=True))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [4]:

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map: 100%|██████████| 307/307 [00:00<00:00, 4822.12 examples/s]
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()



C:\Users\vishn\AppData\Local\Temp\ipykernel_46400\820944952.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,1.043200,0.793214,0.722222,0.656129,0.722222,0.680561
2,0.665900,0.553368,0.813725,0.799203,0.813725,0.799685
3,0.437000,0.433992,0.882353,0.871162,0.882353,0.873982
4,0.308200,0.401867,0.892157,0.888013,0.892157,0.886564


TrainOutput(global_step=1228, training_loss=0.613563997349444, metrics={'train_runtime': 778.5091, 'train_samples_per_second': 12.598, 'train_steps_per_second': 1.577, 'total_flos': 1284799382663424.0, 'train_loss': 0.613563997349444, 'epoch': 4.0})

In [6]:
val_results = trainer.evaluate(val_ds)
test_results = trainer.evaluate(test_ds)

print("Validation Results:", val_results)
print("Test Results:", test_results)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Validation Results: {'eval_loss': 0.4018668830394745, 'eval_accuracy': 0.8921568627450981, 'eval_precision_weighted': 0.8880130546436774, 'eval_recall_weighted': 0.8921568627450981, 'eval_f1_weighted': 0.8865642311219842, 'eval_runtime': 1.2453, 'eval_samples_per_second': 245.718, 'eval_steps_per_second': 31.317, 'epoch': 4.0}
Test Results: {'eval_loss': 0.32073551416397095, 'eval_accuracy': 0.9315960912052117, 'eval_precision_weighted': 0.9207175500751186, 'eval_recall_weighted': 0.9315960912052117, 'eval_f1_weighted': 0.9251208431270548, 'eval_runtime': 1.0424, 'eval_samples_per_second': 294.517, 'eval_steps_per_second': 37.414, 'epoch': 4.0}


('./xlmr_intent_model\\tokenizer_config.json',
 './xlmr_intent_model\\special_tokens_map.json',
 './xlmr_intent_model\\sentencepiece.bpe.model',
 './xlmr_intent_model\\added_tokens.json',
 './xlmr_intent_model\\tokenizer.json')

Model test

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./xlmr_intent_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

def predict_custom_email(subject, body, max_len=256):
    text = f"{subject} [SEP] {body}"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = torch.argmax(probs).item()

    return {
        "intent_type": model.config.id2label[pred_id],
        "confidence": float(probs[pred_id]),
        "all_scores": {
            model.config.id2label[i]: float(probs[i])
            for i in range(len(probs))
        }
    }

subject = "எரிக் மூனுக்கு"
body = """
எரிக் மூனுக்கு -\n\nமாற்றங்களுடன் சிட்டி பேங்க் ப்ரீபேவுக்கான கட்டமைப்பு ஸ்லைடை இணைக்கப்பட்டுள்ளது.\n\nமெலிசா சோலிஸ்\nஎன்ரான் வட அமெரிக்கா\n<கீப்டோக்கன்1>\nW: <PHONE>\nF: <PHONE>
"""

result = predict_custom_email(subject, body)
print(result)

The tokenizer you are loading from './xlmr_intent_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


{'intent_type': 'legitimate', 'confidence': 0.9887266755104065, 'all_scores': {'account_takeover': 0.00043878346332348883, 'credential_harvesting': 0.00048006445285864174, 'data_exfiltration': 0.006697218399494886, 'financial_fraud': 0.0006164402002468705, 'identity_abuse': 0.0009050772641785443, 'infrastructure_exploitation': 0.0004974761977791786, 'legitimate': 0.9887266755104065, 'malware_delivery': 0.0016382792964577675}}
